## Starfysh integration of SpatialVDJ samples

This vignette provides the code using Starfysh to characterize common spatial hubs through integrative spatial deconvolution from multiple samples. The datasets here include 4 distinct ccRCC samples from SpatialVDJ experiment.


In [ ]:
#Import needed modules
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager
from matplotlib import rcParams

import seaborn as sns
sns.set_style('white')

font_list = []
fpaths = matplotlib.font_manager.findSystemFonts()
for i in fpaths:
    try:
        f = matplotlib.font_manager.get_font(i)
        font_list.append(f.family_name)
    except RuntimeError:
        pass

font_list = set(font_list)
plot_font = 'Helvetica' if 'Helvetica' in font_list else 'FreeSans'

rcParams['font.family'] = plot_font
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'figure.figsize': (3,3)})
rcParams.update({'savefig.dpi': 500})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load starfysh
from starfysh import (AA, utils, plot_utils, post_analysis)
from starfysh import starfysh as sf_model

**Create sample meta info**


In [ ]:
meta_info = [['A1','bc2013','Not_provided_Not_relevent','Processed/A1/'],
             ['B1','bc2014','Not_provided_Not_relevent','Processed/B1/'], 
             ['C1','bc2015','Not_provided_Not_relevent','Processed/C1/'],
             ['D1','bc2016','Not_provided_Not_relevent','Processed/D1/'],
            ]
#CID44971_TNBC
meta_info = pd.DataFrame(meta_info,columns=['sample','anonymized_id','RECIST_R_nR','visiumPath'])
# Specify data paths
data_path = 'Processed/' # root data directory
output_path = 'Starfysh_results/'
sample_id2 = 'outs' #name of the folder containing the data
#data_path = '../ToyData/'
#output_path = '../ToyData/Results_integrated/'
sig_file_name = 'Starfysh_results/Signatures_Starfysh3_SMP2.csv' 
print(meta_info)
for sample_id in meta_info['sample']:
    print(sample_id)
    data_path2 = meta_info['visiumPath'][list(meta_info['sample']).index(sample_id)]
    print(data_path2)

In [ ]:
adata_all = []
adata_normed_all = []
img_metadata_all = {}
for sample_id in meta_info['sample']:
    print(sample_id)
    data_path2 = meta_info['visiumPath'][list(meta_info['sample']).index(sample_id)]
    adata, adata_normed = utils.load_adata(data_folder=data_path2,  # root data directory
                                       sample_id=sample_id2,  # sample_id
                                       n_genes=2000  # number of highly variable genes to keep
                                      )
    
    
    adata_all.append(adata)
    adata_normed_all.append(adata_normed)
    
    img_metadata = utils.preprocess_img(data_path2,
                                    sample_id2,
                                    adata_index=adata.obs.index,
                                    hchannel=True #Must be true otherwise POE wont work
                                    )
    #From: https://github.com/azizilab/starfysh/issues/31
    
    adata.obs['anonymized_id']=meta_info['anonymized_id'][list(meta_info['sample']).index(sample_id)]
    adata.obs['RECIST_R_nR']=meta_info['RECIST_R_nR'][list(meta_info['sample']).index(sample_id)]
    adata.obs['patient']=meta_info['sample'][list(meta_info['sample']).index(sample_id)]
    adata.obs['sample']=meta_info['sample'][list(meta_info['sample']).index(sample_id)]
    #sample_id_3 = meta_info['anonymized_id'][list(meta_info['sample']).index(sample_id)]
    adata.obs_names  = adata.obs_names+'-'+sample_id

    adata_normed.obs['anonymized_id']=adata.obs['anonymized_id']
    adata_normed.obs['patient']=adata.obs['patient']
    adata_normed.obs['RECIST_R_nR']=adata.obs['RECIST_R_nR']

    img_metadata['map_info'].index = img_metadata['map_info'].index+'-'+sample_id
    img_metadata_all[sample_id] = img_metadata 

In [ ]:
# Save concat data
import anndata
import scanpy as sc
adata_all = anndata.concat(adata_all)
adata_normed_all = anndata.concat(adata_normed_all)
sc.pp.highly_variable_genes(adata_normed_all)
adata_all.uns = adata_normed_all.uns
adata_all.var = adata_normed_all.var

In [ ]:
#Check
print(adata_all)
print(adata_normed_all)

In [ ]:
# Load signatures
gene_sig = pd.read_csv(os.path.join(sig_file_name))
gene_sig = utils.filter_gene_sig(gene_sig, adata_all.to_df())
gene_sig.head()



In [ ]:
adata_all.obs

In [ ]:
adata_all.obs_names

# Preprocessing: Run archetypes separately to update gene signature

In [ ]:
from starfysh.AA import ArchetypalAnalysis
def assign_archetypes(anchor_df, r=30):
    """
    Assign best 1-1 mapping of archetype community to its closest anchor community (cell-type specific anchor spots)
    Criteria: choose the top cell type in which its anchors belongs to the top r neighbors to the given archetype

    Parameters
    ----------
    anchor_df : pd.DataFrame
        Dataframe of anchor spot indices

`       r : int
        Resolution parameter to threshold archetype - anchor mapping

    Returns
    -------
    map_df : pd.DataFrame
        DataFrame of overlapping spot ratio of each anchor `i` to archetype `j`

    map_dict : dict
        Dictionary of cell type -> mapped archetype
    """
    assert aa_model.arche_df is not None, "Please compute archetypes & assign nearest-neighbors first!"

    n_nbrs, n_archetypes = aa_model.arche_df.shape # number of archetypal spots for each archtype, number of archetypes
    x_concat = np.vstack([aa_model.count, aa_model.archetype])
    anchor_nbrs = anchor_df.values
    archetypal_nbrs = aa_model._get_knns(x_concat, n_nbrs=r, indices=aa_model.n_spots+aa_model.major_idx).T  # r-nearest nbrs to each archetype

    
    
    print(archetypal_nbrs.shape)
    overlaps = np.array(
        [
            [
                len(np.intersect1d(anchor_nbrs[:, i], archetypal_nbrs[:, j]))
                for j in range(archetypal_nbrs.shape[1])
            ]
            for i in range(anchor_nbrs.shape[1])
        ]
    )
    overlaps_df = pd.DataFrame(overlaps, index=anchor_df.columns, columns=aa_model.arche_df.columns)
    arche_argmaxs = overlaps.argmax(0)
    
    distance_df = pd.DataFrame(np.zeros([anchor_df.shape[1],
                                         aa_model.arche_df.shape[1]]), 
                               index=anchor_df.columns, 
                               columns=aa_model.arche_df.columns
                              )
    for i in range(distance_df.shape[0]):
        for j in range(distance_df.shape[1]):
            dist_tmp = 0
            for kk in anchor_df.iloc[:,i]:
                dist_tmp += np.linalg.norm(aa_model.count[kk,:] - 
                                           aa_model.archetype[j,:]
                                          )
            
            distance_df.iloc[i,j] = dist_tmp
    
    map_dict = {}
    for k in range(overlaps.shape[0]):
        list_ = np.argsort(overlaps[k])[::-1]
        for i in list_:
            if (np.argsort(overlaps[:,i])[::-1][0]==k):
                if ((overlaps[k,:]==overlaps[k,i]).sum()==1):
                    map_dict[anchor_df.columns[k]]=aa_model.arche_df.columns[i]
                break 
                
                
    map_dict2 = {}
    for k in range(distance_df.shape[0]):
        list_ = np.argsort(np.array(distance_df)[k])
        #print(list_)
        for i in list_:
            if (np.argsort(np.array(distance_df)[:,i])[0]==k):
                if ((np.array(distance_df)[k,:]==np.array(distance_df)[k,i]).sum()==1):
                    map_dict2[anchor_df.columns[k]]=aa_model.arche_df.columns[i]
                break 
                
    return overlaps_df, distance_df,map_dict,map_dict2


In [ ]:
import importlib
importlib.reload(utils)

gene_sig_all = []
visium_args_all = {}
for sample_id in meta_info['sample']:
    print(sample_id)
    adata = adata_all[adata_all.obs['sample']==sample_id]
    adata_normed = adata_normed_all[adata_normed_all.obs['sample']==sample_id]
    
    sc.pp.highly_variable_genes(adata_normed)
    adata.uns = adata_normed.uns
    adata.var = adata_normed.var

    img_metadata = img_metadata_all[sample_id]
    
    visium_args = utils.VisiumArguments(adata,
                                    adata_normed,
                                    gene_sig,
                                    img_metadata,
                                    window_size=4,
                                    sample_id=sample_id2,#'outs'
                                    n_img_chan = 1,
                                   )
    adata, adata_normed = visium_args.get_adata()
    anchors_df = visium_args.get_anchors()
    aa_model = AA.ArchetypalAnalysis(adata_orig=adata,
                                      # n_neighbors=100  #Atgument not detected
                                    )
    archetype, arche_dict, major_idx, evs = aa_model.compute_archetypes(cn=100, 
                                                                    #r=5,  not used
                                                                    converge=1e-3,
                                                                    display=False)

    arche_df = aa_model.find_archetypal_spots()
    
    markers_df = aa_model.find_markers(n_markers=30, display=False)
    
    percent_anchor = 0.01
    map_df, distance_df,map_dict,map_dict2 = assign_archetypes(anchors_df[:int(percent_anchor*adata.shape[0])], 
                                     r=int(percent_anchor*adata.shape[0])
                                    )  # here we subset the `r` top archetypes
    gene_sig_arch = gene_sig.copy()

    for cell_type in map_dict.keys():
        arch = map_dict[cell_type]
        gene_sig_arch = utils.append_sigs(gene_sig_arch, cell_type, markers_df[arch], n_genes=10)
    gene_sig_all.append(gene_sig_arch)
    visium_args_all[sample_id] = visium_args


In [ ]:
gene_sig

In [ ]:
#gene_sig_all
#Contains reduced signatures for each sample...
for i in range(0, meta_info['anonymized_id'].shape[0]):
    print(i)
    ##Export new signatures...
    tmp_id=meta_info['anonymized_id'][i]
    print(tmp_id)
    gene_temp = gene_sig_all[i]
    gene_temp.to_csv('Starfysh_results/gene_sig_new_'+tmp_id+'_.csv')

### Preprocessing: integrate data
This step is for identifying the anchor spots and providing prior for model training.

utils.VisiumArguments is called for each sample (performed already when updating the signature using archetypes, stored as 'visium_args_all'), followed by utils_integrate.VisiumArguments_integrate for integrating all priors.

Starfysh calcualates cell-type proportion scores for each spot per sample given the annotated signatures (per cell type) as the model's prior

Each spot will be ranked (anchors_df) based on the prior, which represents the rough estimation of the given cell-type's enrichment

In [ ]:
adata_all.obs

In [ ]:
adata_normed_all

In [ ]:
adata_all.obs

In [ ]:
from starfysh import utils_integrate
importlib.reload(utils_integrate)

sc.pp.highly_variable_genes(adata_normed_all)
adata_all.uns = adata_normed_all.uns
adata_all.var = adata_normed_all.var


visium_args = utils_integrate.VisiumArguments_integrate(adata_all,
                                    adata_normed_all,
                                    gene_sig, ##Stay with normal signature
                                    img_metadata_all,
                                    visium_args_all,                                        
                                    window_size=3,
                                    sample_id=meta_info['sample']
                                   )


In [ ]:
import importlib
importlib.reload(sf_model)

from starfysh import utils_integrate
importlib.reload(utils_integrate)

### Processing: Run Starfysh for joint spatial deconvolution & sample integration

**(1) Model parameters**

In [ ]:
n_repeats = 1
epochs = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import importlib
importlib.reload(utils_integrate)

In [ ]:
print(meta_info)

In [ ]:
visium_args.img['A1'].ndim

In [ ]:
visium_args_all

In [ ]:
adata

**(2). Model training**

In [ ]:
model, loss = utils_integrate.run_starfysh(visium_args,
                                 n_repeats=n_repeats,
                                 epochs=epochs,
                                 poe=True, #Means histology to be used...
                                 device=device)

In [ ]:
# Save model
torch.save(model.state_dict(), os.path.join(output_path, 'integ_model.pt'))

In [ ]:
inference_outputs, generative_outputs = sf_model.model_eval_integrate(model,
                                                                      visium_args.adata,
                                                                      visium_args,
                                                                      poe=True,
                                                                      device=device
                                                                     )

In [ ]:
##Extract integrated object to a different object
adata_integrate = visium_args.adata.copy()
adata_integrate.shape

In [ ]:
adata_inte_2 = adata_integrate.copy() #Copy just in case

In [ ]:
#save integrated object
adata_inte_2.write(os.path.join(output_path,'adata_integrated.h5ad'))

In [ ]:
##
# Deconvolution prediction
prop_pred_df = pd.DataFrame(adata_integrate.obsm['qc_m'], index=adata_integrate.obs_names, columns=gene_sig.columns)
prop_pred_df

In [ ]:
##Save deconvolution values
##Export results from deconvolution...
prop_pred_df.to_csv(os.path.join(output_path,'Starfysh_integrated_proportions.csv'), index=True)

## Spatial Hub calculation

In [ ]:
import scanpy.external as sce
adata_integrate.obsm['qz_m'] = np.array(inference_outputs['qz_m'].detach().cpu().numpy())
adata_integrate.obsm['qc_m'] = np.array(inference_outputs['qc_m'].detach().cpu().numpy())
adata_integrate.obsm['X_pca'] = adata_integrate.obsm['qc_m']
adata_integrate.obs[gene_sig.columns]=pd.DataFrame(np.array(adata_integrate.obsm['qc_m']),columns=gene_sig.columns,index=adata_integrate.obs_names)


In [ ]:
sc.pp.neighbors(adata_integrate)
sc.tl.leiden(adata_integrate)


In [ ]:

# Calculate hubs via Phenograph clustering
sce.tl.phenograph(adata_integrate, clustering_algo="louvain",n_jobs=24,
                    resolution_parameter=0.5, ##Reduce resolution, default=1
                  k=130,
                 )


In [ ]:
adata_integrate.obs['hub'] = adata_integrate.obs['pheno_louvain'].astype('category')
adata_integrate.obs['hub']
##23 communities! high heterogeneity

In [ ]:
##Export...
##Export results of hubs
# Deconvolution prediction
hub_pred_df = pd.DataFrame(adata_integrate.obs['hub'], index=adata_integrate.obs_names, columns=['hub'])

hub_pred_df.to_csv(os.path.join(output_path,'Starfysh_integrated_hubs.csv'), index=True)